## Metric definitions

| Column | Source | Meaning |
|---|---|---|
| `cp_active_days` | Copilot usage metrics | Days in the month with any Copilot activity |
| `cp_interactions` | Copilot (`user_initiated_interaction_count`) | Prompts the user explicitly sent (chat/agent); inline completions are not counted |
| `cp_generations` | Copilot (`code_generation_activity_count`) | Times Copilot generated code (inline suggestions, chat/agent output) |
| `cp_acceptances` | Copilot (`code_acceptance_activity_count`) | Generated code the user accepted (Tab accept, apply/insert) |
| `cp_ai_loc_added` | Copilot (`loc_added_sum`) | Lines of code added to the editor by Copilot |
| `cp_ai_loc_deleted` | Copilot (`loc_deleted_sum`) | Lines of code deleted by Copilot |
| `prs_opened` | GitHub pull requests | PRs created by the user in the month |
| `prs_merged` | GitHub pull requests | The user's PRs that were merged |
| `pr_commits` | GitHub pull requests | Total commits across the user's PRs |
| `pr_additions` | GitHub pull requests | Total lines added across the user's PRs |
| `pr_deletions` | GitHub pull requests | Total lines deleted across the user's PRs |


## Load the data
load data and remove some outliers

In [ ]:
import pandas as pd

ai_costs = {
    "2026-08": 9675,
     "2026-09": 13280}
df = pd.read_csv("combined_report_full_anon.csv")
metrics = [c for c in df.columns if c not in ("user_id", "month")]
# print(df.shape)
df.head()

In [ ]:
# Removal of outliers. Set apply_outlier_removal = False to keep all data.
apply_outlier_removal = True
excluded_months = ["2025-12"]  # drop all data for these months
excluded_user_months = [("U058", "2026-08")]  # drop specific (user_id, month) rows

if apply_outlier_removal:
    mask = df["month"].isin(excluded_months)
    for u, m in excluded_user_months:
        mask |= (df["user_id"] == u) & (df["month"] == m)
    print(f"Removing {mask.sum()} of {len(df)} rows")
    df = df[~mask].reset_index(drop=True)

## Add Data

| Column | Source | Meaning |
|---|---|---|
| `calc_pr_net` | calculated | Sum of additions and deletions |
| `calc_delta_per_commit` | calculated | average delta per commit |

In [ ]:
df['calc_pr_net'] = df['pr_additions'] + df['pr_deletions']

df['calc_delta_per_commit'] = df['calc_pr_net'] / df['pr_commits']

# append 'calc_pr_net' and 'calc_delta_per_commit' to metrics

metrics.append('calc_pr_net')
metrics.append('calc_delta_per_commit')

## Trends per metric per month

In [ ]:
import matplotlib.pyplot as plt

for metric in metrics:
    fig, ax = plt.subplots(figsize=(14, 5))
    for month, group in df.groupby("month"):
        values = group[metric].sort_values(ascending=False).reset_index(drop=True)
        ax.plot(values.index + 1, values, label=month)
    ax.set_xlabel("User rank (sorted by " + metric + " within each month)")
    ax.set_ylabel(metric)
    ax.set_title(f"{metric} distribution per user, by month")
    ax.legend()
    plt.show()

In [ ]:
# 3D view: the month axis is depth, so the evolution over time is visible
import numpy as np
from matplotlib import cm

months = sorted(df["month"].unique())
colors = cm.viridis(np.linspace(0, 1, len(months)))

for metric in metrics:
    fig = plt.figure(figsize=(14, 7))
    ax = fig.add_subplot(projection="3d")
    for i, (month, color) in enumerate(zip(months, colors)):
        values = df.loc[df["month"] == month, metric].sort_values(ascending=False).to_numpy()
        ax.plot(np.arange(1, len(values) + 1), np.full(len(values), i), values, color=color)
    ax.set_yticks(range(len(months)), months)
    ax.set_xlabel("User rank")
    ax.set_ylabel("Month")
    ax.set_zlabel(metric)
    ax.set_title(f"{metric} distribution per user, by month")
    ax.view_init(elev=25, azim=-60)
    plt.show()


## Some statistics per metric

In [ ]:
import base64
import io

from IPython.display import HTML, display

for stats_metric in metrics:
    stats = df.groupby("month")[stats_metric].agg(
        ["count", "sum", "mean", "median", "std", "min", "max"]
    ).join(
        df.groupby("month")[stats_metric].quantile([0.25, 0.75]).unstack().add_prefix("q")
    ).round(1)

    panels = [("sum", "sum"), ("mean", "mean"), ("q0.25", "q0.25"), ("q0.75", "q0.75")]
    fig, axes = plt.subplots(2, 2, figsize=(7, 4.5), sharex=True)
    for ax, (col, title) in zip(axes.flat, panels):
        ax.plot(stats.index.astype(str), stats[col], marker="o")
        ax.set_title(title, fontsize=9)
        ax.tick_params(axis="x", rotation=45, labelsize=7)
        ax.tick_params(axis="y", labelsize=7)
        ax.grid(alpha=0.3)
    fig.suptitle(stats_metric, fontsize=10)
    fig.tight_layout()
    buf = io.BytesIO()
    fig.savefig(buf, format="png", dpi=100)
    plt.close(fig)
    img = base64.b64encode(buf.getvalue()).decode()

    display(HTML(
        f"<h4>{stats_metric}</h4>"
        "<div style='display:flex; align-items:flex-start; gap:24px'>"
        f"<div>{stats.to_html()}</div>"
        f"<img src='data:image/png;base64,{img}'/>"
        "</div>"
    ))


## Adoption vs. output (the core question)

- Segment users into AI usage tiers (none, low, high, using quartiles of  cp_interactions  or  cp_ai_loc_added ). Then compare PR throughput ( prs_merged ,  pr_commits ) across the tiers.
- Run correlations or scatter plots between usage ( cp_generations ,  cp_ai_loc_added ) and PR outcomes. Use Spearman, because the data is heavily skewed.
- Compare each user's change from Aug to Sep. Did users whose AI usage grew also see more PRs?

In [ ]:
# Spearman correlation heatmap across all metrics, all months combined
import numpy as np

def plot_corr(data, title, ax):
    corr = data[metrics].corr(method="spearman")
    im = ax.imshow(corr, cmap="coolwarm", vmin=-1, vmax=1)
    ax.set_xticks(range(len(metrics)), metrics, rotation=90)
    ax.set_yticks(range(len(metrics)), metrics)
    for i in range(len(metrics)):
        for j in range(len(metrics)):
            ax.text(j, i, f"{corr.iloc[i, j]:.2f}", ha="center", va="center", fontsize=7)
    ax.set_title(title)
    return im

fig, ax = plt.subplots(figsize=(9, 8))
im = plot_corr(df, "Spearman correlation: all months", ax)
fig.colorbar(im, ax=ax, shrink=0.8)
plt.show()

In [ ]:
# Spearman: AI usage metrics (rows) vs. PR output metrics (columns)
ai_metrics = ["cp_active_days", "cp_interactions", "cp_generations", "cp_acceptances", "cp_ai_loc_added"]
output_metrics = ["prs_opened", "prs_merged", "pr_commits", "pr_additions", "pr_deletions"]

df[ai_metrics + output_metrics].corr(method="spearman").loc[ai_metrics, output_metrics].round(2)

In [ ]:
# Usage tiers: none, then quartiles of non-zero usage, compared on PR output
tier_metric = "cp_interactions"

def assign_tier(data, col):
    tiers = pd.Series("none", index=data.index)
    used = data[col] > 0
    labels = ["low", "medium", "high", "very high"]
    tiers[used] = pd.qcut(data.loc[used, col], 4, labels=labels, duplicates="drop").astype(str)
    return pd.Categorical(tiers, categories=["none"] + labels, ordered=True)

df["tier"] = assign_tier(df, tier_metric)
grouped = df.groupby("tier", observed=True)
tier_stats = grouped[output_metrics].agg(["mean", "median", "min", "max"]).round(1)
tier_stats.insert(0, ("users", "count"), grouped.size())
pd.set_option("display.max_columns", None)
tier_stats

In [ ]:
# Boxplots of PR output per usage tier
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, o in zip(axes, ["prs_merged", "pr_commits", "pr_additions"]):
    present = [t for t in df["tier"].cat.categories if (df["tier"] == t).any()]
    ax.boxplot([df.loc[df["tier"] == t, o] for t in present], tick_labels=present)
    ax.set_yscale("symlog")
    ax.set_title(f"{o} by {tier_metric} tier")
plt.show()

## Quality and efficiency signals

- Acceptance rate:  cp_acceptances / cp_generations  per user. A low rate suggests suggestions aren't useful.
- Merge rate:  prs_merged / prs_opened .
- Commits per PR, and PR size ( pr_additions + pr_deletions ) per PR.
- AI share of code:  cp_ai_loc_added / pr_additions .
- Productivity per active day: PRs or LOC divided by  cp_active_days .

## Population structure

- Count users who are in only one month (101 vs 95 rows), and users with zero AI activity. These are a natural comparison group.
- Look at outliers, such as the user with 3,273 interactions, and rerun the analysis with and without them. Bulk changes of 10k+ lines may be generated or refactoring commits and can skew results.
- A paired Aug-to-Sep comparison (Wilcoxon test) for users who appear in both months.

# TODO 
- Grouping into tiers should probably be per month as some users may have moved into different tiers.
    - This would enable another metric: tier size
- Creating additional data could be interesting - delta-additions (add - del). Average PR size (delta-additions /pr count)
-  

## Treemap of user acitvity

For one specific month, can we create a treemap of user activity using the "cp_interactions" data?

In [ ]:
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

treemap_data = "cp_interactions"
treemap_data = "cp_ai_loc_added"
treemap_data = "prs_merged"
treemap_data = "cp_cost_usd"
treemap_cols = 2  # treemaps per row

treemap_months = [
    m for m in sorted(df["month"].astype(str).unique(), reverse=True)
    if (df[(df["month"].astype(str) == m)][treemap_data] > 0).any()
]
rows = -(-len(treemap_months) // treemap_cols)
fig = make_subplots(
    rows=rows,
    cols=treemap_cols,
    specs=[[{"type": "treemap"}] * treemap_cols for _ in range(rows)],
    subplot_titles=treemap_months,
    vertical_spacing=0.06,
)
for i, treemap_month in enumerate(treemap_months):
    month_df = df[(df["month"].astype(str) == treemap_month) & (df[treemap_data] > 0)]
    sub = px.treemap(
        month_df,
        path=[px.Constant("All users"), "user_id"],
        values=treemap_data,
        color=treemap_data,
    )
    trace = sub.data[0]
    trace.update(
        textinfo="label+value",
        marker=dict(colors=trace.marker.colors, colorscale="Blues", coloraxis=None),
    )
    fig.add_trace(trace, row=i // treemap_cols + 1, col=i % treemap_cols + 1)
fig.update_layout(
    title=f"{treemap_data} per user by month",
    height=400 * rows,
    margin=dict(t=80, l=10, r=10, b=10),
)
fig.show()


## User specific data and analysis

In [ ]:
# user = "U019"
user = "U094"

In [ ]:
# Raw data for the selected user
user_df = df[df["user_id"] == user].sort_values("month")
if user_df.empty:
    raise ValueError(f"No data found for user {user!r}")
user_df

In [ ]:
# One graph per metric for the selected user, over months, in 2x2 grids
months_str = user_df["month"].astype(str)
user_months = sorted(user_df["month"].unique())
month_stats = {
    "mean": df.groupby("month")[metrics].mean(),
    "q0.25": df.groupby("month")[metrics].quantile(0.25),
    "q0.75": df.groupby("month")[metrics].quantile(0.75),
}
for start in range(0, len(metrics), 4):
    fig, axes = plt.subplots(2, 2, figsize=(8, 5))
    for ax, metric in zip(axes.flat, metrics[start:start + 4]):
        ax.plot(months_str, user_df[metric], marker="o", label=user)
        for name, style in [("mean", "--"), ("q0.25", ":"), ("q0.75", ":")]:
            ax.plot(months_str, month_stats[name].loc[user_months, metric], style, label=name)
        ax.set_title(metric, fontsize=9)
        ax.tick_params(labelsize=7)
        ax.grid(alpha=0.3)
        ax.legend(fontsize=6)
    for ax in axes.flat[len(metrics[start:start + 4]):]:
        ax.axis("off")
    fig.suptitle(f"User {user}", fontsize=10)
    fig.tight_layout()
    plt.show()

## Trends for selected metrics and months

Comparing selected months as some months may add more noise as they are non-representative months.

In [ ]:
selected_months = ["2026-01", "2026-09", "2026-10"]
selected_metrics = ["cp_interactions", "prs_merged", "calc_pr_net", "calc_delta_per_commit"]
# Optional y-axis max per metric, e.g. {"prs_merged": 50}; metrics not listed use auto scaling
y_max = {"calc_delta_per_commit": 1000}

for metric in selected_metrics:
    fig, ax = plt.subplots(figsize=(14, 5))
    for month, group in df[df["month"].isin(selected_months)].groupby("month"):
        values = group[metric].sort_values(ascending=False).reset_index(drop=True)
        ax.plot(values.index + 1, values, label=month)
    if y_max.get(metric) is not None:
        ax.set_ylim(bottom=0, top=y_max[metric])
    ax.set_xlabel("User rank (sorted by " + metric + " within each month)")
    ax.set_ylabel(metric)
    ax.set_title(f"{metric} distribution per user, by month")
    ax.legend()
    plt.show()

In [ ]:
# same idea but the diff between two months
selected_months = ["2026-02", "2026-09"]
selected_metrics = ["cp_interactions", "prs_merged", "calc_delta_per_commit"]

# Per-user difference (later month minus earlier month) for users present in both months
start_month, end_month = selected_months
wide = df[df["month"].isin(selected_months)].pivot(index="user_id", columns="month", values=selected_metrics)

for metric in selected_metrics:
    diff = (wide[(metric, end_month)] - wide[(metric, start_month)]).dropna().sort_values(ascending=False).reset_index(drop=True)
    fig, ax = plt.subplots(figsize=(14, 5))
    ax.bar(diff.index + 1, diff, color=np.where(diff >= 0, "tab:green", "tab:red"))
    ax.axhline(0, color="black", linewidth=0.8)
    ax.set_xlabel(f"User rank (sorted by change in {metric}; users present in both months: {len(diff)})")
    ax.set_ylabel(f"{metric} change")
    ax.set_title(f"{metric}: {end_month} minus {start_month} per user (median {diff.median():.1f})")
    plt.show()


# Trying to predict financial gain in introducing claude code

For a month, known budget, known claude code cost, a fixed cutoff for when a user gets claude code
- calculate average interaction per month
- this calculation is based on requests having the average value
- calculate cost per user per month
- assume we can use all pooled resources
- cap costs for users exceeding a value to the claude cost

In [ ]:
month = "2026-10"
budget = 12564 # Oktober
# cp_cost_usd
df_month = df[df["month"] == month].reset_index(drop=True)

c_cost = 18
# c_cost = 90
cutoff = 100

df_month["calc_cost_capped"] = df_month["cp_cost_usd"].where(
    df_month["cp_cost_usd"] <= cutoff, c_cost
)
df_month.head()

In [ ]:
df_month["cp_cost_usd"].agg(["count", "sum", "mean", "median", "std", "min", "max"])

In [ ]:
df_month["calc_cost_capped"].agg(["count", "sum", "mean", "median", "std", "min", "max"])

In [ ]:
cost_before = df_month["cp_cost_usd"].sum()
cost_after = df_month["calc_cost_capped"].sum()
cost_saved = cost_before - cost_after
cost_saved_percent = cost_saved / cost_before * 100
rows_capped = (df_month["cp_cost_usd"] > cutoff).sum()
print(
    "Value saved = {x:.2f} ({y:.2f}%) if {z} people are moved to Claude Code".format(
        x=cost_saved,
        y=cost_saved_percent,
        z=rows_capped,
    )
)